In [1]:
pip install gymnasium[classic_control] numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 771.0 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 285.1 kB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Environment Setup & Random Agent


In [1]:
import gymnasium as gym
import numpy as np
import random

# Create the CartPole environment
# If using older gym: env = gym.make('CartPole-v0')
env = gym.make('CartPole-v1')

print("Environment created successfully!")
print(f"Observation Space: {env.observation_space}")
print(f"Action Space: {env.action_space}")

# --- Run a few episodes with a random agent ---
num_episodes = 5

for episode in range(1, num_episodes + 1):
    state, _ = env.reset()
    done = False
    score = 0
    
    print(f"\n--- Episode {episode} ---")
    
    while not done:
        # env.render() # Uncomment to visualize the agent's actions
        
        # Choose a random action from the action space
        action = env.action_space.sample()
        
        # Take the action and observe the next state, reward, and done status
        next_state, reward, done, truncated, _ = env.step(action)
        
        # Accumulate reward
        score += reward
        
        # Update state
        state = next_state
        
        # Check if the episode is terminated or truncated
        if done or truncated:
            print(f"Episode {episode} finished with score: {score}")
            break

env.close()
print("\nRandom agent exploration complete.")

Environment created successfully!
Observation Space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action Space: Discrete(2)

--- Episode 1 ---
Episode 1 finished with score: 24.0

--- Episode 2 ---
Episode 2 finished with score: 11.0

--- Episode 3 ---
Episode 3 finished with score: 59.0

--- Episode 4 ---
Episode 4 finished with score: 18.0

--- Episode 5 ---
Episode 5 finished with score: 33.0

Random agent exploration complete.


In [ ]:
#Discretizing State Space for Q-Learning

In [2]:
import gymnasium as gym
import numpy as np
import random

# --- Discretization Parameters ---
# Define the number of bins for each state dimension
# These values are often found through experimentation
NUM_BINS = (10, 10, 10, 10) # Example: 10 bins for each of the 4 state variables

# Define the boundaries for each state dimension
# These are based on the typical ranges observed in the CartPole environment
# Cart Position: -4.8 to 4.8
# Cart Velocity: -inf to inf (we'll use a reasonable range)
# Pole Angle: -0.418 rad to 0.418 rad (-24 to 24 degrees)
# Pole Angular Velocity: -inf to inf (we'll use a reasonable range)

# Using bounds that are commonly effective for CartPole
# These bounds might need tuning for different environments or desired performance
BOUNDS = [
    [-4.8, 4.8],      # Cart Position
    [-3.0, 3.0],      # Cart Velocity (approximated)
    [-0.418, 0.418],  # Pole Angle
    [-3.0, 3.0]       # Pole Angular Velocity (approximated)
]

def discretize_state(state):
    """Converts a continuous state into a discrete state tuple."""
    discrete_state_tuple = []
    for i in range(len(state)):
        # Clip the state value to be within the defined bounds
        clipped_value = np.clip(state[i], BOUNDS[i][0], BOUNDS[i][1])
        
        # Calculate the bin index
        # We divide the range of the bound by the number of bins to get the size of each bin
        # Then we scale the clipped value to fit into the bin indices (0 to NUM_BINS[i]-1)
        bin_index = int(np.floor((clipped_value - BOUNDS[i][0]) / (BOUNDS[i][1] - BOUNDS[i][0]) * NUM_BINS[i]))
        
        # Ensure the bin index is within the valid range [0, NUM_BINS[i]-1]
        # This handles potential edge cases due to floating point precision or clipping
        bin_index = max(0, min(bin_index, NUM_BINS[i] - 1))
        
        discrete_state_tuple.append(bin_index)
        
    return tuple(discrete_state_tuple)

# --- Test the discretization function ---
# Create a dummy environment to get observation space details if needed
# For CartPole-v1, observation space is Box([-4.8, -inf, -0.418, -inf], [4.8, inf, 0.418, inf], (4,), float32)
env_test = gym.make('CartPole-v1')

# Example continuous state
# continuous_state = np.array([-0.1, 0.5, 0.02, -0.15])
# For demonstration, let's use the state returned by reset()
initial_state, _ = env_test.reset()

discretized = discretize_state(initial_state)

print(f"Original continuous state: {initial_state}")
print(f"Discretized state: {discretized}")
print(f"Number of bins used: {NUM_BINS}")

env_test.close()

Original continuous state: [-0.04552181  0.04789601  0.04045593  0.04987614]
Discretized state: (4, 5, 5, 5)
Number of bins used: (10, 10, 10, 10)


In [ ]:
#implementing q-learning algorithm

In [3]:

import gymnasium as gym
import numpy as np
import random

# --- Discretization Parameters ---
NUM_BINS = (10, 10, 10, 10) # Example: 10 bins for each of the 4 state variables
BOUNDS = [
    [-4.8, 4.8],      # Cart Position
    [-3.0, 3.0],      # Cart Velocity (approximated)
    [-0.418, 0.418],  # Pole Angle
    [-3.0, 3.0]       # Pole Angular Velocity (approximated)
]

def discretize_state(state):
    """Converts a continuous state into a discrete state tuple."""
    discrete_state_tuple = []
    for i in range(len(state)):
        clipped_value = np.clip(state[i], BOUNDS[i][0], BOUNDS[i][1])
        bin_index = int(np.floor((clipped_value - BOUNDS[i][0]) / (BOUNDS[i][1] - BOUNDS[i][0]) * NUM_BINS[i]))
        bin_index = max(0, min(bin_index, NUM_BINS[i] - 1))
        discrete_state_tuple.append(bin_index)
    return tuple(discrete_state_tuple)

# --- Q-Learning Parameters ---
LEARNING_RATE = 0.1
DISCOUNT_FACTOR = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.995 # Decay rate for epsilon
NUM_EPISODES = 20000 # Number of episodes to train

# --- Initialize Q-table ---
# The Q-table will have dimensions: (num_bins_dim1, num_bins_dim2, ..., num_actions)
# For CartPole, action space is 2 (left, right)
num_actions = gym.make('CartPole-v1').action_space.n

# Initialize Q-table with zeros
# Using a dictionary for sparse Q-tables can be more memory efficient if many states are unreachable
# However, for demonstration with fixed bins, a numpy array is straightforward.
# The shape will be (NUM_BINS[0], NUM_BINS[1], NUM_BINS[2], NUM_BINS[3], num_actions)

# Calculate the total number of discrete states for array initialization
# This can be large, so a dictionary might be better for very large state spaces
# For this example, let's use a dictionary for flexibility

# Q_table = np.zeros(NUM_BINS + (num_actions,))
Q_table = {} # Using a dictionary: key = discrete_state_tuple, value = numpy array of Q-values for actions

def get_q_values(state_tuple):
    """Retrieves Q-values for a given state tuple, initializing if not present."""
    if state_tuple not in Q_table:
        # Initialize Q-values for all actions in this new state
        Q_table[state_tuple] = np.zeros(num_actions)
    return Q_table[state_tuple]

# --- Q-Learning Training Loop ---
env = gym.make('CartPole-v1')

epsilon = EPSILON_START

print(f"Starting Q-Learning training for {NUM_EPISODES} episodes...")

for episode in range(1, NUM_EPISODES + 1):
    state, _ = env.reset()
    discrete_state = discretize_state(state)
    done = False
    score = 0
    
    while not done:
        # Epsilon-greedy action selection
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Explore
        else:
            # Exploit: choose the action with the highest Q-value for the current state
            q_values = get_q_values(discrete_state)
            action = np.argmax(q_values)
            
        # Take the chosen action
        next_state, reward, done, truncated, _ = env.step(action)
        next_discrete_state = discretize_state(next_state)
        
        # Get current Q-value for (state, action)
        current_q = get_q_values(discrete_state)[action]
        
        # Get the maximum Q-value for the next state
        max_next_q = np.max(get_q_values(next_discrete_state))
        
        # Calculate the TD target
        td_target = reward + DISCOUNT_FACTOR * max_next_q
        
        # Calculate the TD error
        td_error = td_target - current_q
        
        # Update the Q-value for the current state-action pair
        # Q(s, a) = Q(s, a) + alpha * (TD Target - Q(s, a))
        Q_table[discrete_state][action] = current_q + LEARNING_RATE * td_error
        
        # Update score and state
        score += reward
        discrete_state = next_discrete_state
        
        if done or truncated:
            break
            
    # Epsilon decay
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    
    # Print progress periodically
    if episode % 1000 == 0:
        print(f"Episode: {episode}/{NUM_EPISODES}, Score: {score:.2f}, Epsilon: {epsilon:.3f}")

env.close()
print("\nQ-Learning training finished.")

# --- Evaluate the trained agent ---
print("\n--- Evaluating Trained Agent ---")

num_eval_episodes = 10

for episode in range(1, num_eval_episodes + 1):
    state, _ = env.reset()
    discrete_state = discretize_state(state)
    done = False
    score = 0
    
    while not done:
        # Choose the best action based on learned Q-values (exploitation only)
        q_values = get_q_values(discrete_state)
        action = np.argmax(q_values)
        
        next_state, reward, done, truncated, _ = env.step(action)
        next_discrete_state = discretize_state(next_state)
        
        score += reward
        discrete_state = next_discrete_state
        
        if done or truncated:
            print(f"Evaluation Episode {episode} finished with score: {score:.2f}")
            break

env.close()
print("\nEvaluation complete.")

# --- Expected Output Guidance ---
print("\nExpected Output Guidance:")
print("During training, you should observe the episode scores gradually increasing.")
print("Epsilon will decay, meaning the agent will explore less and exploit more over time.")
print("After training, the evaluation episodes should consistently achieve higher scores, ideally reaching the maximum possible score (e.g., 500 for CartPole-v1) or close to it.")
print("If scores are not improving, consider adjusting NUM_BINS, LEARNING_RATE, DISCOUNT_FACTOR, or EPSILON_DECAY.")

Starting Q-Learning training for 20000 episodes...
Episode: 1000/20000, Score: 17.00, Epsilon: 0.010
Episode: 2000/20000, Score: 19.00, Epsilon: 0.010
Episode: 3000/20000, Score: 26.00, Epsilon: 0.010
Episode: 4000/20000, Score: 44.00, Epsilon: 0.010
Episode: 5000/20000, Score: 20.00, Epsilon: 0.010
Episode: 6000/20000, Score: 25.00, Epsilon: 0.010
Episode: 7000/20000, Score: 20.00, Epsilon: 0.010
Episode: 8000/20000, Score: 20.00, Epsilon: 0.010
Episode: 9000/20000, Score: 16.00, Epsilon: 0.010
Episode: 10000/20000, Score: 22.00, Epsilon: 0.010
Episode: 11000/20000, Score: 30.00, Epsilon: 0.010
Episode: 12000/20000, Score: 17.00, Epsilon: 0.010
Episode: 13000/20000, Score: 17.00, Epsilon: 0.010
Episode: 14000/20000, Score: 40.00, Epsilon: 0.010
Episode: 15000/20000, Score: 25.00, Epsilon: 0.010
Episode: 16000/20000, Score: 18.00, Epsilon: 0.010
Episode: 17000/20000, Score: 28.00, Epsilon: 0.010
Episode: 18000/20000, Score: 14.00, Epsilon: 0.010
Episode: 19000/20000, Score: 17.00, Epsi

In [4]:
#improving the model : 

In [5]:
import gymnasium as gym
import numpy as np
import random

# --- 1. SHARPENED DISCRETIZATION ---
# We ignore Cart Position/Velocity (1 bin) and focus on Pole Angle/Velocity
NUM_BINS = (1, 1, 6, 12) 
BOUNDS = [
    [-4.8, 4.8],    
    [-3.0, 3.0],    
    [-0.418, 0.418],# Pole Angle (Critical)
    [-3.5, 3.5]     # Pole Angular Velocity (Critical)
]

def discretize_state(state):
    discrete_state = []
    for i in range(len(state)):
        clipped = np.clip(state[i], BOUNDS[i][0], BOUNDS[i][1])
        bin_idx = int(np.floor((clipped - BOUNDS[i][0]) / (BOUNDS[i][1] - BOUNDS[i][0]) * NUM_BINS[i]))
        discrete_state.append(max(0, min(bin_idx, NUM_BINS[i] - 1)))
    return tuple(discrete_state)

# --- 2. THE "PATIENT" HYPERPARAMETERS ---
LEARNING_RATE = 0.1
DISCOUNT_FACTOR = 0.95 # Slightly lower to prioritize keeping it up NOW
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.999 # Slower decay = more thorough learning
NUM_EPISODES = 15000 

# --- 3. REWARD-BASED BRAIN ---
Q_table = {}

def get_q_values(state_tuple, n_actions):
    if state_tuple not in Q_table:
        Q_table[state_tuple] = np.zeros(n_actions)
    return Q_table[state_tuple]

# --- 4. THE OPTIMIZED TRAINING LOOP ---
env = gym.make('CartPole-v1')
epsilon = EPSILON_START
n_actions = env.action_space.n

print("🚀 Training with Optimized Bins and Reward Engineering...")

for episode in range(1, NUM_EPISODES + 1):
    state, _ = env.reset()
    d_state = discretize_state(state)
    done = truncated = False
    score = 0
    
    while not (done or truncated):
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(get_q_values(d_state, n_actions))
            
        next_state, reward, done, truncated, _ = env.step(action)
        
        # REWARD ENGINEERING: 
        # If it fell over, give a massive penalty. If it stayed up, +1.
        final_reward = reward if not (done or truncated) else -200 
        
        next_d_state = discretize_state(next_state)
        
        # TD Error Math
        # $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$
        current_q = get_q_values(d_state, n_actions)[action]
        max_next_q = np.max(get_q_values(next_d_state, n_actions))
        
        # Update logic
        Q_table[d_state][action] += LEARNING_RATE * (final_reward + DISCOUNT_FACTOR * max_next_q - current_q)
        
        d_state = next_d_state
        score += reward

    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    
    if episode % 1000 == 0:
        print(f"Episode {episode} | Last Score: {score} | Epsilon: {epsilon:.3f}")

# --- 5. THE FINAL EXAM (EVALUATION) ---
print("\n🎯 Evaluating the Master Agent (No Randomness allowed)...")
for i in range(5):
    state, _ = env.reset()
    eval_score = 0
    done = truncated = False
    while not (done or truncated):
        d_state = discretize_state(state)
        action = np.argmax(get_q_values(d_state, n_actions))
        state, _, done, truncated, _ = env.step(action)
        eval_score += 1
    print(f"Final Test {i+1} Score: {eval_score}")

env.close()

🚀 Training with Optimized Bins and Reward Engineering...
Episode 1000 | Last Score: 22.0 | Epsilon: 0.368
Episode 2000 | Last Score: 148.0 | Epsilon: 0.135
Episode 3000 | Last Score: 48.0 | Epsilon: 0.050
Episode 4000 | Last Score: 151.0 | Epsilon: 0.018
Episode 5000 | Last Score: 103.0 | Epsilon: 0.010
Episode 6000 | Last Score: 37.0 | Epsilon: 0.010
Episode 7000 | Last Score: 210.0 | Epsilon: 0.010
Episode 8000 | Last Score: 117.0 | Epsilon: 0.010
Episode 9000 | Last Score: 204.0 | Epsilon: 0.010
Episode 10000 | Last Score: 125.0 | Epsilon: 0.010
Episode 11000 | Last Score: 200.0 | Epsilon: 0.010
Episode 12000 | Last Score: 500.0 | Epsilon: 0.010
Episode 13000 | Last Score: 230.0 | Epsilon: 0.010
Episode 14000 | Last Score: 119.0 | Epsilon: 0.010
Episode 15000 | Last Score: 236.0 | Epsilon: 0.010

🎯 Evaluating the Master Agent (No Randomness allowed)...
Final Test 1 Score: 156
Final Test 2 Score: 207
Final Test 3 Score: 165
Final Test 4 Score: 195
Final Test 5 Score: 249


In [ ]:
#final improvement : 

In [6]:
import gymnasium as gym
import numpy as np
import random

# --- 1. SHARPENED DISCRETIZATION ---
NUM_BINS = (1, 1, 6, 12) 
BOUNDS = [
    [-4.8, 4.8],    
    [-3.0, 3.0],    
    [-0.418, 0.418],
    [-3.5, 3.5]     
]

def discretize_state(state):
    discrete_state = []
    for i in range(len(state)):
        clipped = np.clip(state[i], BOUNDS[i][0], BOUNDS[i][1])
        bin_idx = int(np.floor((clipped - BOUNDS[i][0]) / (BOUNDS[i][1] - BOUNDS[i][0]) * NUM_BINS[i]))
        discrete_state.append(max(0, min(bin_idx, NUM_BINS[i] - 1)))
    return tuple(discrete_state)

# --- 2. HYPERPARAMETERS (UPDATED) ---
LEARNING_RATE = 0.1
# UPDATE: Added decay and minimum floor for the Learning Rate
LR_DECAY = 0.9995  
LR_MIN = 0.01

DISCOUNT_FACTOR = 0.95 
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.999 
NUM_EPISODES = 15000 

# --- 3. REWARD-BASED BRAIN ---
Q_table = {}

def get_q_values(state_tuple, n_actions):
    if state_tuple not in Q_table:
        Q_table[state_tuple] = np.zeros(n_actions)
    return Q_table[state_tuple]

# --- 4. THE OPTIMIZED TRAINING LOOP ---
env = gym.make('CartPole-v1')
epsilon = EPSILON_START
n_actions = env.action_space.n

print("🚀 Training with Optimized Bins, Reward Engineering, and LR Decay...")

for episode in range(1, NUM_EPISODES + 1):
    state, _ = env.reset()
    d_state = discretize_state(state)
    done = truncated = False
    score = 0
    
    while not (done or truncated):
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(get_q_values(d_state, n_actions))
            
        next_state, reward, done, truncated, _ = env.step(action)
        
        # Penalize failure heavily
        final_reward = reward if not (done or truncated) else -200 
        
        next_d_state = discretize_state(next_state)
        
        # TD Error Math
        # $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$
        current_q = get_q_values(d_state, n_actions)[action]
        max_next_q = np.max(get_q_values(next_d_state, n_actions))
        
        # UPDATE: The update now uses the progressively smaller LEARNING_RATE
        Q_table[d_state][action] += LEARNING_RATE * (final_reward + DISCOUNT_FACTOR * max_next_q - current_q)
        
        d_state = next_d_state
        score += reward

    # UPDATE: Decay both Epsilon (curiosity) and Learning Rate (knowledge sensitivity)
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    LEARNING_RATE = max(LR_MIN, LEARNING_RATE * LR_DECAY)
    
    if episode % 1000 == 0:
        # Added LR to the print statement to monitor its progress
        print(f"Ep {episode} | Score: {score} | Epsilon: {epsilon:.3f} | LR: {LEARNING_RATE:.4f}")

# --- 5. THE FINAL EXAM ---
print("\n🎯 Evaluating the Master Agent...")
for i in range(5):
    state, _ = env.reset()
    eval_score = 0
    done = truncated = False
    while not (done or truncated):
        d_state = discretize_state(state)
        action = np.argmax(get_q_values(d_state, n_actions))
        state, _, done, truncated, _ = env.step(action)
        eval_score += 1
    print(f"Final Test {i+1} Score: {eval_score}")

env.close()

🚀 Training with Optimized Bins, Reward Engineering, and LR Decay...
Ep 1000 | Score: 16.0 | Epsilon: 0.368 | LR: 0.0606
Ep 2000 | Score: 149.0 | Epsilon: 0.135 | LR: 0.0368
Ep 3000 | Score: 224.0 | Epsilon: 0.050 | LR: 0.0223
Ep 4000 | Score: 357.0 | Epsilon: 0.018 | LR: 0.0135
Ep 5000 | Score: 256.0 | Epsilon: 0.010 | LR: 0.0100
Ep 6000 | Score: 59.0 | Epsilon: 0.010 | LR: 0.0100
Ep 7000 | Score: 500.0 | Epsilon: 0.010 | LR: 0.0100
Ep 8000 | Score: 160.0 | Epsilon: 0.010 | LR: 0.0100
Ep 9000 | Score: 395.0 | Epsilon: 0.010 | LR: 0.0100
Ep 10000 | Score: 500.0 | Epsilon: 0.010 | LR: 0.0100
Ep 11000 | Score: 500.0 | Epsilon: 0.010 | LR: 0.0100
Ep 12000 | Score: 21.0 | Epsilon: 0.010 | LR: 0.0100
Ep 13000 | Score: 443.0 | Epsilon: 0.010 | LR: 0.0100
Ep 14000 | Score: 115.0 | Epsilon: 0.010 | LR: 0.0100
Ep 15000 | Score: 211.0 | Epsilon: 0.010 | LR: 0.0100

🎯 Evaluating the Master Agent...
Final Test 1 Score: 225
Final Test 2 Score: 217
Final Test 3 Score: 244
Final Test 4 Score: 248
Fina